# Suite RCH — Reachable

`Reachable.of(root)` returns the root and every object reachable through adjacencies, in first-reference order. It is
schema-agnostic, so most cases use hand-written `Fake` visitables that no schema describes.

In [ ]:
import { Proxies, Reachable, Schemas as S } from "@mbse/schemas/Framework";
import { AttributeError, NotImplementedError } from "@mbse/schemas/Framework/Errors";
import type { Visitable } from "@mbse/schemas/Framework/Visitors";
import { assert, Fake, raises, same, text } from "./support.js";

const names = (objs: Visitable[]) => objs.map((o) => (o as Fake).values.get("n"));
const node = (n: string) => new Fake("Anything", { n });
function link(obj: Fake, ...targets: Fake[]): void {
  const list = obj.adjacencies.get("out") ?? [];
  list.push(...targets.map((t) => ({ to: t })));
  obj.adjacencies.set("out", list);
}

## RCH-01 · A lone object reaches only itself

In [ ]:
{
  const solo = node("solo");
  assert(same(Reachable.of(solo), [solo]));
}

## RCH-02 · Breadth-first, first-reference order

In [ ]:
{
  const [a, b, c, d, e] = ["a", "b", "c", "d", "e"].map(node) as [Fake, Fake, Fake, Fake, Fake];
  link(a, b, c);
  link(b, d);
  link(c, e);
  assert(same(names(Reachable.of(a)), ["a", "b", "c", "d", "e"]));
  assert(same(names(Reachable.of(b)), ["b", "d"])); // adjacency direction as written by accept()
}

## RCH-03 · Cycles, self-loops and diamonds terminate and list each object once

In [ ]:
{
  const [p, q, r] = ["p", "q", "r"].map(node) as [Fake, Fake, Fake];
  link(p, q);
  link(q, r);
  link(r, p);
  link(p, p);
  assert(same(names(Reachable.of(p)), ["p", "q", "r"]));
  const [top, left, right, bottom] = ["top", "left", "right", "bottom"].map(node) as [Fake, Fake, Fake, Fake];
  link(top, left, right);
  link(left, bottom);
  link(right, bottom);
  assert(same(names(Reachable.of(top)), ["top", "left", "right", "bottom"]));
}

## RCH-04 · Several adjacencies, several links per entry, and properties are ignored

In [ ]:
{
  const [hub, x, y, z] = ["hub", "x", "y", "z"].map(node) as [Fake, Fake, Fake, Fake];
  hub.adjacencies.set("trio", [{ first: x, second: y, weight: 1.5, tag: "t" }]);
  hub.adjacencies.set("other", [{ to: z, blob: new Uint8Array([0]) }]);
  assert(same(names(Reachable.of(hub)), ["hub", "x", "y", "z"]));
}

## RCH-05 · Identity decides sameness, not object identity or equality

In [ ]:
{
  const twin1 = new Fake("T", { n: "t1" }, {}, "same");
  const twin2 = new Fake("T", { n: "t2" }, {}, "same");
  const root = node("root");
  link(root, twin1, twin2);
  assert(same(names(Reachable.of(root)), ["root", "t1"])); // the second object with the same identity is not revisited
  const eq1 = node("eq");
  const eq2 = node("eq");
  link(root, eq1, eq2);
  assert(Reachable.of(root).length === 4); // equal content but distinct identities are distinct objects
}

## RCH-06 · The root is always first, even when others link back to it

In [ ]:
{
  const back = node("back");
  const start = node("start");
  link(start, back);
  link(back, start);
  assert(Reachable.of(start)[0] === start && Reachable.of(back)[0] === back);
}

## RCH-07 · Long chains do not recurse

In [ ]:
{
  const chain = Array.from({ length: 20_000 }, (_, i) => node(String(i)));
  for (let i = 0; i + 1 < chain.length; i++) link(chain[i] as Fake, chain[i + 1] as Fake);
  assert(Reachable.of(chain[0] as Fake).length === 20_000);
}

## RCH-08 · Errors thrown by `accept` propagate

In [ ]:
{
  class Broken extends Fake {
    override accept(): void {
      throw new Error("broken accept");
    }
  }
  raises(Error, () => Reachable.of(new Broken("B")), "broken accept");
  const root2 = node("root2");
  link(root2, new Broken("B"));
  raises(Error, () => Reachable.of(root2));
}

## RCH-09 · The collector refuses operations it cannot answer

In [ ]:
{
  const collector = new Reachable._Collector(() => null);
  raises(NotImplementedError, () => collector.adjacency("x", (a) => a.me()));
  raises(NotImplementedError, () => collector.adjacency("x", (a) => a.add((e) => e.link("l", (k) => k.target(() => null)))));
  raises(AttributeError, () => collector.property("x", (p) => p.value((v) => v.as_native((n) => n.get()))));
}

## RCH-10 · Proxies: the same component from every member

In [ ]:
{
  const Rel = new S.OfRelation.Builder().links("a", "b").create();
  const Node = new S.OfObject.Builder().properties(text("n")).relations((r) => r.name("out").of(Rel).me("a"), (r) => r.name("in").of(Rel).me("b")).create();
  Proxies.register("Node", Node);
  Proxies.register("Rel", Rel);
  const B = Proxies.Builders;
  const n1 = B.Node().n("1").create();
  const n2 = B.Node().n("2").out((x: any) => x.b(n1)).create();
  const n3 = B.Node().n("3").out((x: any) => x.b(n2)).create();
  const loner = B.Node().n("loner").create();
  const ids = (o: Visitable) => new Set(Reachable.of(o).map((v) => v.identity()));
  const component = ids(n1);
  assert(same([...component].sort(), [...ids(n2)].sort()) && same([...component].sort(), [...ids(n3)].sort()));
  assert(component.size === 3 && same(Reachable.of(loner), [loner]));
}